# Phase 2

In [ ]:
!pip install -q codecarbon transformers accelerate sentencepiece pillow pandas pymupdf

In [ ]:
import os
import re
import csv
import json
import time
from datetime import datetime

import pandas as pd
import torch
import fitz  # PyMuPDF
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import EmissionsTracker
from huggingface_hub import login

login("hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# =========================
# CONFIG
# =========================
#LLM_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"      # faster
#LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
#LLM_MODEL= "Qwen/Qwen2.5-14B-Instruct"
#LLM_MODEL= "mistralai/Ministral-3-14B-Instruct-2512"

NUM_RUNS = 1
CPU_WARMUP_SECS = 10
GPU_WARMUP_SECS = 10
COOLDOWN_SECS = 60
MEASURE_POWER_SECS = 1

ALL_PAGES_MAX = 10
CHUNK_SIZE = 6

PDF_DIR = "/content/drive/MyDrive/All_Req_Pdfs"

MAIN_OUTPUT_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"
MODEL_TAG = LLM_MODEL.replace("/", "__")
PHASE2_ROOT = os.path.join(MAIN_OUTPUT_DIR, "phase2_runs", MODEL_TAG)

os.makedirs(PHASE2_ROOT, exist_ok=True)

In [ ]:
def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=MEASURE_POWER_SECS,
        log_level="warning",
    )

print("PDF_DIR:", PDF_DIR)
print("PHASE2_ROOT:", PHASE2_ROOT)

In [ ]:
def prepare_tokenizer_and_model(tokenizer, model):
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        elif tokenizer.eos_token_id is not None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    if hasattr(model, "generation_config"):
        if getattr(model.generation_config, "pad_token_id", None) is None:
            model.generation_config.pad_token_id = tokenizer.pad_token_id

    return tokenizer, model

In [ ]:
setup_tracking_dir = os.path.join(PHASE2_ROOT, "setup_tracking")

setup_tracker = make_tracker(
    output_dir=setup_tracking_dir,
    project_name="phase2_setup",
    output_file="phase2_setup_emissions.csv"
)

print("PHASE 2 SETUP TRACKER START")
setup_tracker.start()
setup_t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

tokenizer, model = prepare_tokenizer_and_model(tokenizer, model)

model.eval()

setup_emissions = setup_tracker.stop()
setup_duration = time.time() - setup_t0
print("PHASE 2 SETUP TRACKER STOP")

setup_summary = {
    "phase": "phase2",
    "model_id": LLM_MODEL,
    "setup_duration_sec": setup_duration,
    "setup_emissions_kg": setup_emissions,
    "device": device,
    "input_pdf_dir": PDF_DIR
}
save_json(setup_summary, os.path.join(setup_tracking_dir, "phase2_setup_summary.json"))

print("Loaded LLM:", LLM_MODEL)
setup_summary

In [ ]:
def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_phase2_llm_once():
    prompt = "Read this requirement text and summarize in one short line."

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    _ = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

def run_phase2_warmup():
    warmup_cpu(CPU_WARMUP_SECS)

    print(f"GPU/LLM warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()
    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_phase2_llm_once()
        time.sleep(2)
    print("GPU/LLM warm-up finished.\n")

print("\n========== PHASE 2 WARM-UP START (EXCLUDED) ==========")
run_phase2_warmup()
print("========== PHASE 2 WARM-UP STOP ==========\n")

In [ ]:
def pdf_to_page_texts(pdf_path: str):
    doc = fitz.open(pdf_path)
    page_texts = []
    for i in range(len(doc)):
        text = doc[i].get_text("text")
        page_texts.append(text.strip())
    doc.close()
    return page_texts

SYSTEM_PROMPT = """
You are a Senior QA Engineer.

Input: A Requirements PDF for ONE software project.
The PDF may include text and tables.

GOAL:
Generate test cases ONLY from what is explicitly stated in the PDF about the SOFTWARE.

ABSOLUTE RULES (STRICT):
1) Do NOT create test cases about the DOCUMENT itself.
   Do NOT test: cover page, table of contents, glossary, references, appendix, formatting, section numbering.

2) Do NOT assume missing test steps, missing validations, missing expected results.
   If the PDF does not explicitly specify enough information to write a valid test case,
   SKIP that test case completely. Also do not assume test steps if it is not mentioned
   explicitly in the PDF. Only Write those informations which are mentioned in the PDF.

3) Every test case MUST be directly supported by the PDF content.
   Do not add new features, new flows, or best-practice cases unless explicitly written.

OUTPUT RULES:
- Output PLAIN TEXT only (no JSON)
- Do NOT output markdown (no **bold**, no headings like ###, no bullet lists)
- Do NOT output placeholders like <...>, "string", "..."
- Do NOT repeat these instructions
""".strip()

USER_PROMPT_TEMPLATE = """
Read the provided PDF pages text for ONE project.

Task:
1) Write TEST CASES based ONLY on explicit requirements written in the PDF.
   If a requirement is too vague to produce valid steps/expected result, SKIP it.

OUTPUT FORMAT (STRICT):

PROJECT OVERVIEW:
(10–12 lines max)

----------------------------
TEST CASES
----------------------------

For EACH test case, use EXACTLY this structure:

Test Case ID: TC-<MODULE>-<NUMBER>
Title:
Module:
Source Pages:
Preconditions:
Test Data:
Test Steps:
Expected Result:

Rules:
- This is the desired output format. If there is no information for a particular field keep it blank/None
- Use sequential numbering: 001, 002, 003...
- Make Module a meaningful word (no placeholders)
- Source Pages must be numbers like: 1 or 2,3 (PDF page numbers)
- Test Steps must be numbered 1., 2., 3....
- DO NOT add any test cases about the PDF document structure
""".strip()

def build_prompt(page_text_block, user_text):
    return f"""<|system|>
{SYSTEM_PROMPT}
<|user|>
{user_text}

PDF CONTENT:
------------------------
{page_text_block}
------------------------
<|assistant|>
"""

@torch.no_grad()
def run_llm(page_text_block, user_text, max_new_tokens=2400):
    prompt = build_prompt(page_text_block, user_text)

    input_token_count = len(tokenizer.encode(prompt, add_special_tokens=False))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    if "<|assistant|>" in decoded:
        decoded = decoded.split("<|assistant|>", 1)[-1].strip()

    output_token_count = len(tokenizer.encode(decoded, add_special_tokens=False))

    return decoded, input_token_count, output_token_count

def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield i, lst[i:i+chunk_size]

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def clean_line(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^#+\s*", "", s)
    s = re.sub(r"^[-•]\s*", "", s)
    s = re.sub(r"^\*+\s*", "", s)
    s = re.sub(r"\*+", "", s)
    s = s.replace("Source UML Pages:", "Source Pages:")
    return s.strip()

def is_document_level_case(title: str, module: str) -> bool:
    t = normalize_ws(title).lower()
    m = normalize_ws(module).lower()

    bad_keywords = [
        "document", "table of contents", "glossary", "references", "appendix",
        "structure", "formatting", "section", "chapter", "index", "toc"
    ]
    bad_modules = {"introduction", "glossary", "references", "appendix", "appendices"}

    if m in bad_modules:
        return True
    if any(k in t for k in bad_keywords):
        return True
    return False

def parse_testcases_from_text(text: str):
    lines = [clean_line(ln) for ln in (text or "").splitlines()]
    rows = []
    current = None
    field = None

    header_map = {
        "Test Case ID:": "Test Case ID",
        "Title:": "Title",
        "Module:": "Module",
        "Source Pages:": "Source Pages",
        "Preconditions:": "Preconditions",
        "Test Data:": "Test Data",
        "Test Steps:": "Test Steps",
        "Expected Result:": "Expected Result",
    }

    def flush():
        nonlocal current
        if not current:
            return

        required = list(header_map.values())
        for k in required:
            if not normalize_ws(current.get(k, "")):
                current = None
                return

        blob = " ".join(normalize_ws(current.get(k, "")).lower() for k in required)
        if any(x in blob for x in ["tc-<", "<module>", "<number>", "string", "..."]):
            current = None
            return

        if is_document_level_case(current.get("Title",""), current.get("Module","")):
            current = None
            return

        rows.append(current)
        current = None

    for ln in lines:
        s = ln.strip()
        if not s:
            continue

        if s.startswith("Test Case ID:"):
            flush()
            tcid = s.replace("Test Case ID:", "").strip()
            current = {v: "" for v in header_map.values()}
            current["Test Case ID"] = tcid
            field = None
            continue

        if not current:
            continue

        matched = False
        for prefix, key in header_map.items():
            if s.startswith(prefix):
                matched = True
                field = key
                current[key] = s[len(prefix):].strip()
                break
        if matched:
            continue

        if field:
            current[field] = (current[field] + "\n" + s).strip() if current[field] else s

    flush()
    return rows

def dedupe_rows(rows):
    seen = set()
    out = []
    for r in rows:
        sig = (normalize_ws(r["Module"]).lower(), normalize_ws(r["Title"]).lower())
        if sig in seen:
            continue
        seen.add(sig)
        out.append(r)
    return out

def renumber(rows):
    for idx, r in enumerate(rows, start=1):
        module = re.sub(r"[^A-Za-z0-9]+", "", r["Module"])[:20] or "MOD"
        r["Test Case ID"] = f"TC-{module}-{idx:03d}"
    return rows

In [ ]:
pdf_files = sorted([
    os.path.join(PDF_DIR, f)
    for f in os.listdir(PDF_DIR)
    if f.lower().endswith(".pdf")
])

print("Total PDFs found:", len(pdf_files))
print("First few:", [os.path.basename(x) for x in pdf_files[:5]])

In [ ]:
def run_phase2_single(run_id: int):
    run_root = os.path.join(PHASE2_ROOT, f"run_{run_id}")
    os.makedirs(run_root, exist_ok=True)

    OUT_DIR = os.path.join(run_root, "REQ_LLM_SINGLE_MODIFICATION")
    TRACKING_DIR = os.path.join(run_root, "inference_tracking")
    TOKEN_CSV = os.path.join(run_root, "phase2_token_counts.csv")

    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    tracker = make_tracker(
        output_dir=TRACKING_DIR,
        project_name="phase2_inference",
        output_file=f"phase2_run_{run_id}_inference_emissions.csv"
    )

    print(f"\nPHASE 2 RUN {run_id} INFERENCE TRACKER START")
    tracker.start()
    infer_t0 = time.time()

    master_dfs = []
    run_summary = []
    token_rows = []

    for idx, PDF_PATH in enumerate(pdf_files, start=1):
        try:
            print(f"\n[{idx}/{len(pdf_files)}] Processing:", os.path.basename(PDF_PATH))

            pages = pdf_to_page_texts(PDF_PATH)
            print("Total pages:", len(pages))

            base = os.path.splitext(os.path.basename(PDF_PATH))[0]
            raw_txt_path = os.path.join(OUT_DIR, f"{base}_raw_output.txt")
            csv_path = os.path.join(OUT_DIR, f"{base}_testcases.csv")

            all_outputs = []
            all_rows = []

            if len(pages) <= ALL_PAGES_MAX:
                print(f"Running ALL pages at once: 1..{len(pages)}")
                page_text_block = "\n\n".join(
                    [f"[Page {i+1}]\n{pages[i]}" for i in range(len(pages))]
                )
                user_text = USER_PROMPT_TEMPLATE + f"\n\nPDF PAGES INCLUDED: 1..{len(pages)}\n"
                out_text, in_tok, out_tok = run_llm(page_text_block, user_text, max_new_tokens=2600)
                all_outputs.append(out_text)
                all_rows.extend(parse_testcases_from_text(out_text))

                token_rows.append({
                    "phase": "phase2",
                    "run_id": run_id,
                    "pdf_file": os.path.basename(PDF_PATH),
                    "chunk_type": "all_pages",
                    "pages": f"1..{len(pages)}",
                    "input_tokens": in_tok,
                    "output_tokens": out_tok,
                    "total_tokens": (in_tok or 0) + (out_tok or 0),
                })
            else:
                print(f"Running in chunks of {CHUNK_SIZE} pages (auto fallback)")
                for chunk_start, page_chunk in chunk_list(pages, CHUNK_SIZE):
                    page_nums = list(range(chunk_start + 1, chunk_start + 1 + len(page_chunk)))
                    print(f"Chunk pages: {page_nums}")

                    page_text_block = "\n\n".join(
                        [f"[Page {page_nums[i]}]\n{page_chunk[i]}" for i in range(len(page_chunk))]
                    )

                    user_text = USER_PROMPT_TEMPLATE + f"\n\nPDF PAGES IN THIS CHUNK: {','.join(map(str,page_nums))}\n"
                    out_text, in_tok, out_tok = run_llm(page_text_block, user_text, max_new_tokens=2600)

                    all_outputs.append(out_text)
                    all_rows.extend(parse_testcases_from_text(out_text))

                    token_rows.append({
                        "phase": "phase2",
                        "run_id": run_id,
                        "pdf_file": os.path.basename(PDF_PATH),
                        "chunk_type": "chunk",
                        "pages": ",".join(map(str, page_nums)),
                        "input_tokens": in_tok,
                        "output_tokens": out_tok,
                        "total_tokens": (in_tok or 0) + (out_tok or 0),
                    })

            with open(raw_txt_path, "w", encoding="utf-8") as f:
                f.write(("\n\n" + "="*80 + "\n\n").join(all_outputs))

            rows = dedupe_rows(all_rows)
            rows = renumber(rows)
            df = pd.DataFrame(rows)

            df.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)

            print("✅ Saved raw TXT:", raw_txt_path)
            print("✅ Saved CSV:", csv_path)
            print("Parsed testcases:", len(df))

            df.insert(0, "PDF File", os.path.basename(PDF_PATH))
            master_dfs.append(df)

            run_summary.append({
                "PDF File": os.path.basename(PDF_PATH),
                "Pages": len(pages),
                "Parsed testcases": len(df),
                "CSV": csv_path,
                "RAW": raw_txt_path,
                "Status": "OK"
            })

        except Exception as e:
            print("❌ Failed:", os.path.basename(PDF_PATH), "|", type(e).__name__, str(e)[:200])
            run_summary.append({
                "PDF File": os.path.basename(PDF_PATH),
                "Pages": "",
                "Parsed testcases": 0,
                "CSV": "",
                "RAW": "",
                "Status": f"FAILED: {type(e).__name__}: {str(e)[:200]}"
            })

    if master_dfs:
        master_df = pd.concat(master_dfs, ignore_index=True)
    else:
        master_df = pd.DataFrame(columns=[
            "PDF File", "Test Case ID", "Title", "Module", "Source Pages",
            "Preconditions", "Test Data", "Test Steps", "Expected Result"
        ])

    master_csv_path = os.path.join(OUT_DIR, "ALL_PDFS_testcases_master.csv")
    master_df.to_csv(master_csv_path, index=False, quoting=csv.QUOTE_ALL)
    print("\n✅ Saved MASTER CSV:", master_csv_path)
    print("Total master testcases:", len(master_df))

    summary_df = pd.DataFrame(run_summary)
    summary_csv_path = os.path.join(OUT_DIR, "RUN_SUMMARY.csv")
    summary_df.to_csv(summary_csv_path, index=False, quoting=csv.QUOTE_ALL)
    print("✅ Saved RUN SUMMARY:", summary_csv_path)

    token_df = pd.DataFrame(token_rows)
    token_df.to_csv(TOKEN_CSV, index=False)

    inference_emissions = tracker.stop()
    inference_duration = time.time() - infer_t0
    print(f"PHASE 2 RUN {run_id} INFERENCE TRACKER STOP")

    inference_summary = {
        "phase": "phase2",
        "run_id": run_id,
        "mode": "repeated_runs",
        "model_id": LLM_MODEL,
        "input_pdf_dir": PDF_DIR,
        "output_root": OUT_DIR,
        "master_csv": master_csv_path,
        "summary_csv": summary_csv_path,
        "token_csv": TOKEN_CSV,
        "inference_duration_sec": inference_duration,
        "inference_emissions_kg": inference_emissions,
    }

    save_json(
        inference_summary,
        os.path.join(TRACKING_DIR, f"phase2_run_{run_id}_inference_summary.json")
    )

    return inference_summary

In [ ]:
all_run_summaries = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n==================== PHASE 2 RUN {run_id}/{NUM_RUNS} ====================")
    run_summary = run_phase2_single(run_id)
    all_run_summaries.append(run_summary)

    if run_id < NUM_RUNS:
        print(f"\nCooldown for {COOLDOWN_SECS} seconds...\n")
        time.sleep(COOLDOWN_SECS)